# 🍛 Food Detection Version 3 Training (221 Categories + Multi-Dish Thali & Banana Leaf Plates)
### South Indian & Global Food Inspection AI Pipeline (V3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dharmenthiran-spi/SouthIndian_Food_Nutrition/blob/main/Food_Detection/NoteBooks/Food%20Detection/Version3/food_detection_v3_training.ipynb)

---
### 🚀 What's New in Version 3:
- **Base Dataset**: All 11,050 single-dish images across **221 food categories** from Version 2.
- **Real Multi-Dish Plates**: 500 downloaded Thali, Banana Leaf, and Breakfast Combo images with multi-object YOLO bounding boxes.
- **Synthetic Combo Plates**: 600 stitched 4-dish plate images with exact ground-truth bounding boxes.
- **Transfer Learning**: Fine-tunes from `food_detection_v2_best.pt` (if present) or `yolov8s.pt` with `mosaic=1.0` & `mixup=0.1`.
- **Real-Time Google Drive Auto-Save + Step 5B Resume**: Every checkpoint saves directly to your Google Drive.

## Step 1: Install Dependencies, Verify GPU & Mount Google Drive
Mounts Google Drive upfront so all checkpoints and `best.pt` save in real-time.

In [ ]:
!pip install -q ultralytics torchvision opencv-python-headless pyyaml matplotlib seaborn pandas onnx

import torch, sys, os, ultralytics
print('=' * 60)
print('🖥️  SYSTEM & HARDWARE SPECIFICATIONS (V3)')
print('=' * 60)
print(f'Python Version   : {sys.version.split()[0]}')
print(f'PyTorch Version  : {torch.__version__}')
print(f'Ultralytics      : {ultralytics.__version__}')
print(f'CUDA Available   : {"✅ Yes" if torch.cuda.is_available() else "❌ No (Switch Colab runtime to GPU!)"}')
if torch.cuda.is_available():
    print(f'GPU Device Name  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM Available   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
print('=' * 60)
ultralytics.checks()

if 'google.colab' in str(get_ipython()):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('✅ Google Drive connected! All training epochs will save directly to Drive.')
    except Exception as e:
        print(f'Drive note: {e}')

## Step 2: Fresh GitHub Repository Clone
Clones the repository into Colab's fast local NVMe storage (`/content/`).

In [ ]:
import os

REPO_URL = 'https://github.com/Dharmenthiran-spi/SouthIndian_Food_Nutrition.git'
REPO_NAME = 'SouthIndian_Food_Nutrition'

if 'google.colab' in str(get_ipython()):
    if not os.path.exists(REPO_NAME):
        print(f'🚀 Freshly cloning {REPO_NAME} from GitHub...')
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    print(f'✅ Active Working Directory: {os.getcwd()}')
else:
    print(f'Running in local workspace: {os.getcwd()}')

## Step 3: Version 3 Dataset Configuration (221 Classes + Multi-Dish)
Configures absolute paths in `Food_Detection/DataSet/Version3 Used Data/dataset.yaml`.

In [ ]:
import os, yaml, glob

candidate_yamls = [
    'Food_Detection/DataSet/Version3 Used Data/dataset.yaml',
    os.path.join(os.getcwd(), 'Food_Detection', 'DataSet', 'Version3 Used Data', 'dataset.yaml'),
    '/content/SouthIndian_Food_Nutrition/Food_Detection/DataSet/Version3 Used Data/dataset.yaml'
]

dataset_yaml_path = next((os.path.abspath(p) for p in candidate_yamls if os.path.exists(p)), None)
if not dataset_yaml_path:
    raise FileNotFoundError('❌ Version 3 dataset.yaml not found!')

dataset_dir = os.path.dirname(dataset_yaml_path)
with open(dataset_yaml_path, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['path'] = dataset_dir
data_cfg['train'] = 'images/train'
data_cfg['val'] = 'images/val'
data_cfg['test'] = 'images/test'

with open(dataset_yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_cfg, f, sort_keys=False)

names = data_cfg.get('names', {})
n_tr = len(glob.glob(os.path.join(dataset_dir, 'images', 'train', '*.jpg')))
n_va = len(glob.glob(os.path.join(dataset_dir, 'images', 'val', '*.jpg')))
n_te = len(glob.glob(os.path.join(dataset_dir, 'images', 'test', '*.jpg')))

print(f'✅ Version 3 dataset.yaml : {dataset_yaml_path}')
print(f'🔥 Total Classes (nc)    : {len(names)}')
print(f'📸 Train / Val / Test    : {n_tr} / {n_va} / {n_te} (Total: {n_tr + n_va + n_te})')

## Step 4: Multi-Dish Sample Visual Inspection
Previews multi-dish & synthetic combo plate samples with ground-truth bounding boxes.

In [ ]:
import glob, cv2, matplotlib.pyplot as plt

train_img_dir = os.path.join(dataset_dir, 'images', 'train')
train_lbl_dir = os.path.join(dataset_dir, 'labels', 'train')
multi_samples = glob.glob(os.path.join(train_img_dir, 'v3_*.jpg'))[:4]
if not multi_samples:
    multi_samples = glob.glob(os.path.join(train_img_dir, '*.jpg'))[:4]

if multi_samples:
    fig, axes = plt.subplots(1, len(multi_samples), figsize=(20, 5))
    if len(multi_samples) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, multi_samples):
        img = cv2.imread(img_path)
        h, w, _ = img.shape
        stem = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(train_lbl_dir, f'{stem}.txt')
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as lf:
                for line in lf:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cid = int(parts[0])
                        xc, yc, bw, bh = map(float, parts[1:5])
                        x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
                        x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
                        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                        cname = names.get(cid, str(cid))
                        cv2.putText(img, cname, (x1, max(y1 - 6, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(stem[:28], fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## Step 5: Model Training Options (Version 3)
Choose **Step 5A** for fresh Version 3 training (starts from Version 2 `food_detection_v2_best.pt` if available), or **Step 5B** to resume an interrupted run.

### Step 5A: Launch Version 3 Training (100 Epochs, Multi-Dish Optimized)

In [ ]:
from ultralytics import YOLO
import os

EPOCHS = 100
BATCH_SIZE = 16
IMAGE_SIZE = 640
RUN_NAME = 'food_detection_v3_run'

# Prefer fine-tuning from Version 2 trained weights if present
v2_candidates = [
    'Food_Detection/Models/Food Detection/Version2/food_detection_v2_best.pt',
    '/content/drive/MyDrive/Food_Inspection_V2_Models/Food_Detection/food_detection_v2_run/weights/best.pt'
]
MODEL_BASE = next((p for p in v2_candidates if os.path.exists(p)), 'yolov8s.pt')

if os.path.exists('/content/drive/MyDrive'):
    PROJECT_DIR = '/content/drive/MyDrive/Food_Inspection_V3_Models/Food_Detection'
    print(f'💾 Saving checkpoints in real-time to Google Drive:\n   {PROJECT_DIR}/{RUN_NAME}')
else:
    PROJECT_DIR = 'Food_Detection/runs/train'
    print(f'📁 Saving checkpoints locally: {PROJECT_DIR}/{RUN_NAME}')

os.makedirs(PROJECT_DIR, exist_ok=True)
print('=' * 65)
print(f'🚀 LAUNCHING FOOD DETECTION VERSION 3 (Base: {MODEL_BASE})')
print('=' * 65)

model = YOLO(MODEL_BASE)
results = model.train(
    data=dataset_yaml_path,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0 if torch.cuda.is_available() else 'cpu',
    project=PROJECT_DIR,
    name=RUN_NAME,
    patience=15,
    mosaic=1.0,
    mixup=0.1,
    save=True,
    save_period=5,
    exist_ok=True,
    plots=True,
    verbose=True
)
print('\n✅ Version 3 Training Completed!')

### Step 5B: 🔄 Resume Interrupted Version 3 Training
Run this cell if Colab disconnected mid-training to continue from `last.pt`.

In [ ]:
from ultralytics import YOLO
import os

if 'google.colab' in str(get_ipython()):
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print(e)

PROJECT_DIR = '/content/drive/MyDrive/Food_Inspection_V3_Models/Food_Detection' if os.path.exists('/content/drive/MyDrive') else 'Food_Detection/runs/train'
RUN_NAME = 'food_detection_v3_run'
last_ckpt = os.path.join(PROJECT_DIR, RUN_NAME, 'weights', 'last.pt')

if os.path.exists(last_ckpt):
    print(f'🔄 Resuming from {last_ckpt}...')
    resume_model = YOLO(last_ckpt)
    results = resume_model.train(resume=True)
else:
    print('❌ No last.pt checkpoint found. Run Step 5A first.')

## Step 6: Analytical Plots & Metric Visualization

In [ ]:
import os, glob, matplotlib.pyplot as plt, matplotlib.image as mpimg

if 'PROJECT_DIR' not in globals():
    PROJECT_DIR = '/content/drive/MyDrive/Food_Inspection_V3_Models/Food_Detection' if os.path.exists('/content/drive/MyDrive') else 'Food_Detection/runs/train'
if 'RUN_NAME' not in globals():
    RUN_NAME = 'food_detection_v3_run'
run_dir = os.path.join(PROJECT_DIR, RUN_NAME)

for title, fname, sz in [
    ('YOLOv8 V3 Training Curves', 'results.png', (18, 9)),
    ('YOLOv8 V3 Confusion Matrix', 'confusion_matrix.png', (16, 16)),
]:
    p = os.path.join(run_dir, fname)
    if os.path.exists(p):
        plt.figure(figsize=sz)
        plt.imshow(mpimg.imread(p))
        plt.title(title, fontsize=15)
        plt.axis('off')
        plt.show()

## Step 7 & 8: Export `food_detection_v3_best.pt` / `.onnx` & 1-Click Download

In [ ]:
import os, shutil
from ultralytics import YOLO

if 'PROJECT_DIR' not in globals():
    PROJECT_DIR = '/content/drive/MyDrive/Food_Inspection_V3_Models/Food_Detection' if os.path.exists('/content/drive/MyDrive') else 'Food_Detection/runs/train'
if 'RUN_NAME' not in globals():
    RUN_NAME = 'food_detection_v3_run'
run_dir = os.path.join(PROJECT_DIR, RUN_NAME)

best_src = os.path.join(run_dir, 'weights', 'best.pt')
out_dir = 'Food_Detection/Models/Food Detection/Version3'
os.makedirs(out_dir, exist_ok=True)

if os.path.exists(best_src):
    v3_pt = os.path.join(out_dir, 'food_detection_v3_best.pt')
    shutil.copy2(best_src, v3_pt)
    shutil.copy2(best_src, 'food_detection_v3_best.pt')
    print(f'🏆 Saved: {v3_pt}')
    try:
        onnx_f = YOLO(v3_pt).export(format='onnx', imgsz=640, simplify=True)
        if os.path.exists(onnx_f):
            shutil.copy2(onnx_f, os.path.join(out_dir, 'food_detection_v3_best.onnx'))
            shutil.copy2(onnx_f, 'food_detection_v3_best.onnx')
    except Exception as e:
        print(f'ONNX note: {e}')

    try:
        from google.colab import files
        files.download('food_detection_v3_best.pt')
        if os.path.exists('food_detection_v3_best.onnx'):
            files.download('food_detection_v3_best.onnx')
    except ImportError:
        pass
else:
    print('❌ best.pt not found yet.')